<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/HIDROCARBUROS/PP_Produccion_Diaria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Producción fiscalizada DIARIA por lote (PERUPETRO)**: líquidos en barriles y gas natural en Mcf y MMBtu. Ejecuta el Bloque 1 y luego el Bloque 2. Resultados en Drive: `DATA/PP_produccion_diaria.csv` y `CONTROL/CONTROL PP produccion_diaria.csv`.

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN Y LECTOR (ejecutar una vez por sesión) ═══
!pip install pdfplumber requests certifi cryptography pandas google-api-python-client --quiet
import os, io, re, time, html, statistics, unicodedata
from datetime import date
from urllib.parse import urljoin, unquote
import requests
import pandas as pd
import pdfplumber
from IPython.display import display
print("✅ Librerías listas")

# ── Lector de los «Reportes de producción fiscalizada» DIARIOS (tabla día × lote, leída por coordenadas) ──

MESES_ES = ["ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO", "JULIO", "AGOSTO", "SETIEMBRE", "OCTUBRE",
            "NOVIEMBRE", "DICIEMBRE"]
NUMERO = re.compile(r"^-?\d{1,3}(,\d{3})*(\.\d+)?$|^-?\d+(\.\d+)?$")
LOTE_TXT = re.compile(r"^(\d{1,3}(-?[A-Z]{1,2})?|[IVXL]+|Z-\d{1,2}[A-Z]?|\d{1,3}_LIC|192_LIC)\.?$")
NO_OPERADOR = {"LOTE", "DIA", "SUB", "TOTAL", "MCF", "MMBTU", "M3", "M3/D", ":", "LOTE:", "MES", "ANO", "SEPTIEMBRE"} | set(MESES_ES)


def _sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFD", str(t)) if unicodedata.category(c) != "Mn").upper()


def lote_diario(t):
    """'X.' → 'X', '192_LIC' → '192', 'Z-2B' → 'Z-2B', 'I (TT)' → 'I(TT)'."""
    t = t.upper().replace(" ", "").rstrip(".")
    t = re.sub(r"_LIC$", "", t)
    return t


def _lineas(ws, tol=2.6):
    out = []
    for w in sorted(ws, key=lambda w: (w["top"], w["x0"])):
        if out and abs(out[-1][0] - w["top"]) <= tol:
            out[-1][1].append(w)
        else:
            out.append([w["top"], [w]])
    return [(t, sorted(l, key=lambda w: w["x0"])) for t, l in out]


def _cabecera(pg):
    """Título, unidad, año y mes de la página."""
    t = _sin_tildes(pg.extract_text() or "")
    tit = re.search(r"REPORTE DE PRODUCCION FISCALIZADA DE (HIDROCARBUROS LIQUIDOS|GAS NATURAL)[^(\n]*\(([^)]+)\)", t)
    anio = re.search(r"ANO\s+(\d{4})", t)
    mes = re.search(r"MES\s+(" + "|".join(MESES_ES + ["SEPTIEMBRE"]) + r")", t)
    rep = re.search(r"REPORTE AL\s*:\s*(\d{2}/\d{2}/\d{4})", t)
    if not (tit and anio and mes):
        return None
    unidad = tit.group(2).strip().upper().replace("BARRILES", "BBL").replace("M3/D", "M3")
    m = mes.group(1).replace("SEPTIEMBRE", "SETIEMBRE")
    return {"FLUIDO": "LIQUIDOS" if "LIQUIDOS" in tit.group(1) else "GAS NATURAL", "UNIDAD": unidad,
            "AÑO": int(anio.group(1)), "MES": MESES_ES.index(m) + 1, "REPORTE_AL": rep.group(1) if rep else ""}


def _num(t):
    return float(t.replace(",", ""))


def leer_pagina(pg):
    cab = _cabecera(pg)
    if not cab:
        return None
    ws = pg.extract_words()
    lineas = _lineas(ws)
    # 1. fila de lotes: la primera línea con 5+ códigos de lote (en gas van como "Lote : 31-C")
    fila_lotes = None
    for top, lw in lineas:
        cols = []
        for i, w in enumerate(lw):
            t = w["text"].upper()
            if LOTE_TXT.match(t) and not (i and lw[i - 1]["text"].upper() in ("AÑO", "ANO")):
                x0, x1, txt = w["x0"], w["x1"], w["text"]
                if i + 1 < len(lw) and re.fullmatch(r"\(T+\)", lw[i + 1]["text"]):     # "I (TT)"
                    x1, txt = lw[i + 1]["x1"], txt + lw[i + 1]["text"]
                if cab["FLUIDO"] == "GAS NATURAL" and i >= 2 and lw[i - 2]["text"].upper() == "LOTE":
                    x0 = lw[i - 2]["x0"]                     # el rótulo "Lote : 31-C" completo está centrado
                cols.append({"LOTE_ORIGINAL": txt, "LOTE": lote_diario(txt), "c": (x0 + x1) / 2, "top": top})
        if len(cols) >= 5:
            fila_lotes = (top, cols)
            break
    if not fila_lotes:
        return None
    top_l, cols = fila_lotes
    # 2. columnas SUB TOTAL / TOTAL (en la cabecera, hasta 20 pt arriba de la fila de lotes)
    tot = [w for w in ws if w["text"].upper() == "TOTAL" and top_l - 20 <= w["top"] <= top_l + 3
           and w["x0"] > min(c["c"] for c in cols)]
    subs = [w for w in ws if w["text"].upper() == "SUB" and top_l - 22 <= w["top"] <= top_l]
    for w in tot:
        es_sub = any(abs((s["x0"] + s["x1"]) / 2 - (w["x0"] + w["x1"]) / 2) < 15 and s["top"] < w["top"] for s in subs)
        cols.append({"LOTE_ORIGINAL": "SUB TOTAL" if es_sub else "TOTAL", "LOTE": "SUBTOTAL" if es_sub else "TOTAL",
                     "c": (w["x0"] + w["x1"]) / 2, "top": w["top"]})
    cols.sort(key=lambda c: c["c"])
    dif = [b["c"] - a["c"] for a, b in zip(cols, cols[1:])]
    ancho = statistics.median(dif) if dif else 30
    # 3. producto de cada columna: en líquidos, antes del primer SUB TOTAL = petróleo y condensado; luego LGN
    grupo = "PETROLEO Y CONDENSADO" if cab["FLUIDO"] == "LIQUIDOS" else "GAS NATURAL"
    for c in cols:
        c["PRODUCTO"] = grupo
        if c["LOTE"] == "SUBTOTAL" and grupo == "PETROLEO Y CONDENSADO":
            grupo = "LGN"
    # 4. operadores: palabras de la cabecera agrupadas en frases (un mismo operador puede abarcar varios lotes)
    cand = [w for w in ws if top_l - 14 <= w["top"] <= top_l + 20 and abs(w["top"] - top_l) > 2.5
            and not NUMERO.match(w["text"]) and _sin_tildes(w["text"]) not in NO_OPERADOR
            and not re.fullmatch(r"(19|20)\d\d", w["text"])
            and not LOTE_TXT.match(w["text"].upper())]
    frases = []
    for top, lw in _lineas(cand):
        for w in lw:
            txt = re.sub(r"MCF|MMBTU", "", w["text"], flags=re.I)        # "ENERMcfGIA" → "ENERGIA"
            if not txt:
                continue
            if frases and frases[-1]["top"] == top and w["x0"] - frases[-1]["x1"] < 4:
                frases[-1].update(x1=w["x1"], txt=frases[-1]["txt"] + " " + txt)
            else:
                frases.append({"top": top, "x0": w["x0"], "x1": w["x1"], "txt": txt})
    # una frase en el renglón de abajo que se superpone a otra de arriba la completa ("CORPORACIO" + "N DEL SUR")
    finales = []
    for f in sorted(frases, key=lambda f: f["top"]):
        arriba = [g for g in finales if g["top"] < f["top"] and f["x0"] < g["x1"] and f["x1"] > g["x0"]]
        if arriba:
            g = arriba[-1]
            g["txt"] = g["txt"] + ("" if g["txt"].endswith(("CORPORACIO",)) else " ") + f["txt"]
            g["x0"], g["x1"] = min(g["x0"], f["x0"]), max(g["x1"], f["x1"])
        else:
            finales.append(dict(f))
    for c in cols:
        if c["LOTE"] in ("SUBTOTAL", "TOTAL"):
            c["OPERADOR"] = ""
            continue
        dentro = [f for f in finales if f["x0"] - ancho / 2 <= c["c"] <= f["x1"] + ancho / 2]
        c["OPERADOR"] = min(dentro, key=lambda f: abs((f["x0"] + f["x1"]) / 2 - c["c"]))["txt"] if dentro else ""
    # 5. filas de días: los valores van alineados a la derecha de su celda
    bordes = [c["c"] + ancho / 2 for c in cols]
    filas = []
    x_min = cols[0]["c"] - ancho / 2
    for top, lw in lineas:
        if top <= top_l + 3:
            continue
        txt0 = _sin_tildes(lw[0]["text"])
        if txt0.startswith(("ACUMULADO", "TOTAL", "PROMEDIO", "OBSERVACIONES")):
            break
        dia = next((w for w in lw if w["x1"] < x_min + 2 and re.fullmatch(r"\d{1,2}", w["text"])), None)
        nums = [w for w in lw if w["x1"] >= x_min and NUMERO.match(w["text"])]
        if not nums:
            continue
        if dia is None:                       # en gas el n.º de día queda en otra línea, 1 pt más abajo
            dia = next((w for t2, l2 in lineas if 0 < abs(t2 - top) <= 3 for w in l2
                        if w["x1"] < x_min + 2 and re.fullmatch(r"\d{1,2}", w["text"])), None)
        if dia is None or not 1 <= int(dia["text"]) <= 31:
            continue
        valores = {}
        for w in nums:
            k = min(range(len(cols)), key=lambda i: abs(bordes[i] - w["x1"]))
            if abs(bordes[k] - w["x1"]) <= ancho / 2:
                valores[k] = _num(w["text"])
        filas.append({"DIA": int(dia["text"]), "valores": valores})
    return {**cab, "columnas": cols, "dias": filas}


def extraer_diario(path):
    """Lista de páginas leídas (un PDF de gas trae Mcf y MMBtu en páginas distintas)."""
    with pdfplumber.open(path) as pdf:
        return [p for p in (leer_pagina(pg) for pg in pdf.pages) if p]


def a_filas(pag, fuente=""):
    """Filas largas (día × lote) y control de cada día (suma de lotes vs SUB TOTAL / TOTAL del PDF)."""
    cols, filas, ctrl = pag["columnas"], [], []
    tol = 1.0 if pag["UNIDAD"] == "BBL" else 0.05
    for d in pag["dias"]:
        v = d["valores"]
        fecha = f"{d['DIA']:02d}/{pag['MES']:02d}/{pag['AÑO']}"
        lotes = [i for i, c in enumerate(cols) if c["LOTE"] not in ("SUBTOTAL", "TOTAL")]
        for i in lotes:
            c = cols[i]
            filas.append({"FECHA": fecha, "AÑO": pag["AÑO"], "MES": pag["MES"], "DIA": d["DIA"],
                          "FLUIDO": pag["FLUIDO"], "PRODUCTO": c["PRODUCTO"], "UNIDAD": pag["UNIDAD"],
                          "OPERADOR": c["OPERADOR"], "LOTE_ORIGINAL": c["LOTE_ORIGINAL"], "LOTE": c["LOTE"],
                          "VALOR": v.get(i, 0.0), "FUENTE": fuente})
        # control: cada SUB TOTAL contra los lotes de su producto; el TOTAL contra todos los lotes
        dif_max = 0.0
        for i, c in enumerate(cols):
            if c["LOTE"] == "SUBTOTAL" and i in v:
                s = sum(v.get(j, 0.0) for j in lotes if cols[j]["PRODUCTO"] == c["PRODUCTO"])
                dif_max = max(dif_max, abs(s - v[i]))
            if c["LOTE"] == "TOTAL" and i in v:
                dif_max = max(dif_max, abs(sum(v.get(j, 0.0) for j in lotes) - v[i]))
        ctrl.append({"FECHA": fecha, "FLUIDO": pag["FLUIDO"], "UNIDAD": pag["UNIDAD"],
                     "CELDAS": len(v), "DIF_MAX": round(dif_max, 4), "ESTADO": "OK" if dif_max <= tol else "REVISAR"})
    return filas, ctrl


In [ ]:
# ═══ BLOQUE 2 · DESCARGA, LECTURA, CONTROL Y DRIVE ═══
# Descarga los reportes diarios de la página «Producción Diaria» de PERUPETRO (líquidos en barriles y gas natural en
# Mcf y MMBtu), los lee, valida cada día contra el TOTAL / SUB TOTAL del PDF y acumula la serie en Drive.

# ── 1. CONFIGURACIÓN FIJA ────────────────────────────────────────────────────
DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
PAGINA_DIARIA = ("https://www.perupetro.com.pe/wps/portal/corporativo/PerupetroSite/estadisticas/"
                 "producci%C3%B3n%20hidrocarburos/producci%C3%B3n%20diaria/!ut/p/z1/04_Sj9CPykssy0xPLMnMz0vMAfIjo8zi_Yx"
                 "cTTw8TAy93AN8LQwCTUJcvEKADCdfI_1wsAIDHMDRQD8Kt35DHxOofhwKAo3MKNIPUhBFjPvxKIjC771w_ShCSqLwucHM3xS_AnAY4"
                 "FHgbm5krF-QGxoaGmGQ6ZmuqAgAQ52_bw!!/dz/d5/L2dBISEvZ0FBIS9nQSEh/")
BASE = "https://www.perupetro.com.pe"
PAUSA = 1                                         # segundos entre descargas
ruta = "/content/PerupetroDiaria"; os.makedirs(ruta, exist_ok=True)
VERIFICAR_SSL = True     # False solo como último recurso si la cadena no se puede completar (desactiva la verificación)
CAB = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                     "Chrome/124.0 Safari/537.36"}
NOMBRE_DATOS = "PP_produccion_diaria.csv"
NOMBRE_CONTROL = "CONTROL PP produccion_diaria.csv"

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
import logging
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name,mimeType)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def subir(ruta_local, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_local)
    for i in range(1, 6):
        try:
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
            else:
                f = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                         fields="id", supportsAllDrives=True).execute()
                existentes[nombre] = f["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/5: {e}"); time.sleep(2 ** i)
    return False

def bajar_de_drive(file_id, destino):
    with open(destino, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
        hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()

datos_drive = archivos_en(DRIVE_FOLDER_ID)
control_drive = archivos_en(DRIVE_FOLDER_CONTROL_ID)
if "PDF_DIARIA" not in datos_drive:
    datos_drive["PDF_DIARIA"] = drive.files().create(body={"name": "PDF_DIARIA", "parents": [DRIVE_FOLDER_ID],
                                                           "mimeType": "application/vnd.google-apps.folder"},
                                                     fields="id", supportsAllDrives=True).execute()["id"]
PDF_FOLDER_ID = datos_drive["PDF_DIARIA"]
pdf_drive = archivos_en(PDF_FOLDER_ID)

# Departamento de cada lote, del mapa de lotes (DATA/PP_LOTES_GEO.csv), si ya existe
DEPTO_LOTE = {}
if "PP_LOTES_GEO.csv" in datos_drive:
    try:
        bajar_de_drive(datos_drive["PP_LOTES_GEO.csv"], os.path.join(ruta, "PP_LOTES_GEO.csv"))
        _g = pd.read_csv(os.path.join(ruta, "PP_LOTES_GEO.csv"), encoding="utf-8-sig", dtype=str)
        DEPTO_LOTE = {k: v for k, v in zip(_g["LOTE"], _g["DEPARTAMENTO"]) if isinstance(v, str) and v}
    except Exception as e:
        print(f"⚠️ No se pudo leer PP_LOTES_GEO.csv ({e})")
print(f"☁️  Drive conectado · reportes diarios ya guardados: {len(pdf_drive)}")

# ── 3. CERTIFICADO HTTPS DE PERUPETRO (el servidor no envía su certificado intermedio) ──
import ssl, socket, certifi
from cryptography import x509
from cryptography.hazmat.primitives.serialization import Encoding
from cryptography.x509.oid import ExtensionOID, AuthorityInformationAccessOID

def cert_del_servidor(host):
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    with socket.create_connection((host, 443), timeout=30) as s, ctx.wrap_socket(s, server_hostname=host) as ss:
        return x509.load_der_x509_certificate(ss.getpeercert(binary_form=True))

def emisores(cert):
    try:
        aia = cert.extensions.get_extension_for_oid(ExtensionOID.AUTHORITY_INFORMATION_ACCESS).value
    except x509.ExtensionNotFound:
        return []
    return [d.access_location.value for d in aia if d.access_method == AuthorityInformationAccessOID.CA_ISSUERS]

def leer_cert(datos):
    try:
        return x509.load_der_x509_certificate(datos)
    except ValueError:
        return x509.load_pem_x509_certificate(datos)

def bundle_con_intermedios(host, destino):
    cert, pems = cert_del_servidor(host), []
    for _ in range(3):
        urls = emisores(cert)
        if not urls or cert.issuer == cert.subject:
            break
        cert = leer_cert(requests.get(urls[0], timeout=30).content)
        pems.append(cert.public_bytes(Encoding.PEM).decode())
    with open(destino, "w") as f:
        f.write(open(certifi.where()).read() + "\n" + "\n".join(pems))
    return destino, len(pems)

def preparar_verificacion():
    try:
        requests.get(BASE, headers=CAB, timeout=30)
        return True
    except requests.exceptions.SSLError:
        pass
    try:
        ruta_ca, n = bundle_con_intermedios("www.perupetro.com.pe", os.path.join(ruta, "ca_perupetro.pem"))
        requests.get(BASE, headers=CAB, timeout=30, verify=ruta_ca)
        print(f"🔒 Certificado de PERUPETRO completado con {n} intermedio(s): conexión verificada")
        return ruta_ca
    except Exception as e:
        if VERIFICAR_SSL:
            print(f"⚠️ No se pudo completar el certificado de PERUPETRO ({type(e).__name__}: {e}).\n"
                  "   Opciones: sube los PDF a mano a DATA/PDF_DIARIA, o pon VERIFICAR_SSL = False arriba (último recurso).")
            return True
        print("⚠️ VERIFICAR_SSL = False: se descarga SIN verificar el certificado (solo PDF públicos)")
        return False

VERIFY = preparar_verificacion()

# ── 4. ENLACES DE LA PÁGINA «PRODUCCIÓN DIARIA» ─────────────────────────────
def tipo_enlace(href, texto):
    """LIQUIDOS o GAS (Mcf; trae también la hoja en MMBtu). El de m3 repite el gas en otra unidad y se omite."""
    t = _sin_tildes(html.unescape(re.sub(r"<[^>]+>", " ", texto)) + " " + unquote(href).replace("+", " "))
    if "LIQUIDO" in t:
        return "LIQUIDOS"
    if "GAS" in t and not re.search(r"M3", t):
        return "GAS"
    return None

try:
    pagina = requests.get(PAGINA_DIARIA, headers=CAB, timeout=60, verify=VERIFY).text
except Exception as e:
    pagina = ""
    print(f"⚠️ No se pudo leer la página de PERUPETRO ({e}). Se usarán solo los PDF que ya estén en Drive/PDF_DIARIA.")
enlaces = {}
for href, texto in re.findall(r'<a[^>]+href="([^"]+)"[^>]*>(.*?)</a>', pagina, re.S | re.I):
    href = html.unescape(href)
    if "/wps/wcm/connect/" in href or href.lower().endswith(".pdf"):
        tipo = tipo_enlace(href, texto)
        if tipo:
            enlaces.setdefault(urljoin(BASE, href), tipo)
print(f"🔗 Reportes diarios en la página: {sum(1 for t in enlaces.values() if t == 'LIQUIDOS')} de líquidos y "
      f"{sum(1 for t in enlaces.values() if t == 'GAS')} de gas")

def descargar(url):
    for i in range(1, 4):
        try:
            r = requests.get(url, headers=CAB, timeout=120, verify=VERIFY)
            if r.status_code == 200 and r.content[:4] == b"%PDF":
                return r.content
            print(f"   ↻ HTTP {r.status_code} ({i}/3)")
        except Exception as e:
            print(f"   ↻ {type(e).__name__} ({i}/3)")
        time.sleep(3 * i)
    return None

# ── 5. DESCARGA, LECTURA Y CONTROL ──────────────────────────────────────────
# Cada mes se guarda como PP_DIARIA_<LIQUIDOS|GAS>_<año>_<mes>.pdf; el del mes en curso se reemplaza en cada corrida
# (PERUPETRO lo actualiza día a día). Los PDF subidos a mano a DATA/PDF_DIARIA también se leen.
leidos, control = {}, []

def procesar(local, origen):
    try:
        paginas = extraer_diario(local)
    except Exception as e:
        control.append({"ARCHIVO": origen, "ESTADO": f"ERROR: {type(e).__name__}: {e}"}); return None
    if not paginas:
        control.append({"ARCHIVO": origen, "ESTADO": "NO ES REPORTE DIARIO"}); return None
    p0 = paginas[0]
    nombre = f"PP_DIARIA_{'LIQUIDOS' if p0['FLUIDO'] == 'LIQUIDOS' else 'GAS'}_{p0['AÑO']}_{p0['MES']:02d}.pdf"
    for pag in paginas:
        filas, ctrl = a_filas(pag, nombre)
        clave = (pag["FLUIDO"], pag["UNIDAD"], pag["AÑO"], pag["MES"])
        if clave in leidos and leidos[clave]["REPORTE_AL"] >= pag["REPORTE_AL"][6:] + pag["REPORTE_AL"][3:5] + pag["REPORTE_AL"][:2]:
            continue                               # ya hay uno igual o más reciente
        c = pd.DataFrame(ctrl)
        leidos[clave] = {"REPORTE_AL": pag["REPORTE_AL"][6:] + pag["REPORTE_AL"][3:5] + pag["REPORTE_AL"][:2], "filas": filas,
                         "ctrl": {"ARCHIVO": nombre, "FLUIDO": pag["FLUIDO"], "UNIDAD": pag["UNIDAD"], "AÑO": pag["AÑO"],
                                  "MES": pag["MES"], "REPORTE_AL": pag["REPORTE_AL"], "DIAS": len(c),
                                  "DIAS_OK": int((c["ESTADO"] == "OK").sum()) if len(c) else 0,
                                  "DIF_MAX": float(c["DIF_MAX"].max()) if len(c) else None,
                                  "LOTES": len({f["LOTE"] for f in filas}),
                                  "ESTADO": "OK" if len(c) and (c["ESTADO"] == "OK").all() else "REVISAR"}}
    return nombre

for n, (url, tipo) in enumerate(enlaces.items(), 1):
    contenido = descargar(url)
    if contenido is None:
        control.append({"ARCHIVO": url, "ESTADO": "ERROR DE DESCARGA"}); print(f"❌ [{n}] {url[:90]}"); continue
    local = os.path.join(ruta, f"tmp_{n}.pdf"); open(local, "wb").write(contenido)
    nombre = procesar(local, url)
    if nombre:
        os.replace(local, os.path.join(ruta, nombre))
        subir(os.path.join(ruta, nombre), PDF_FOLDER_ID, pdf_drive, mime="application/pdf")
        print(f"✅ [{n}/{len(enlaces)}] {nombre}")
    time.sleep(PAUSA)
# PDF ya guardados (o subidos a mano) que no vinieron en esta corrida
nuevos = {c["ctrl"]["ARCHIVO"] for c in leidos.values()}
for nombre_d, fid in pdf_drive.items():
    if nombre_d.lower().endswith(".pdf") and nombre_d not in nuevos:
        local = os.path.join(ruta, nombre_d); bajar_de_drive(fid, local); procesar(local, nombre_d)

# ── 6. SERIE ACUMULADA EN DRIVE ──────────────────────────────────────────────
nuevas = pd.DataFrame([f for v in leidos.values() for f in v["filas"]])
if len(nuevas):
    nuevas["DEPARTAMENTO"] = nuevas["LOTE"].map(lambda l: DEPTO_LOTE.get(l, DEPTO_LOTE.get(re.sub(r"\(T+\)$", "", l), "")))
previa = pd.DataFrame()
if NOMBRE_DATOS in datos_drive:
    bajar_de_drive(datos_drive[NOMBRE_DATOS], os.path.join(ruta, NOMBRE_DATOS))
    previa = pd.read_csv(os.path.join(ruta, NOMBRE_DATOS), encoding="utf-8-sig", dtype={"LOTE": str, "LOTE_ORIGINAL": str})
if len(previa) and len(nuevas):                   # los meses leídos ahora reemplazan a los guardados
    k_nuevas = set(zip(nuevas["FLUIDO"], nuevas["UNIDAD"], nuevas["AÑO"], nuevas["MES"]))
    previa = previa[[k not in k_nuevas for k in zip(previa["FLUIDO"], previa["UNIDAD"], previa["AÑO"], previa["MES"])]]
diaria = pd.concat([previa, nuevas], ignore_index=True)
if len(diaria):
    diaria["_f"] = pd.to_datetime(diaria["FECHA"], format="%d/%m/%Y")
    diaria = diaria.sort_values(["_f", "FLUIDO", "UNIDAD", "PRODUCTO", "LOTE"]).drop(columns="_f")
    diaria.to_csv(os.path.join(ruta, NOMBRE_DATOS), index=False, encoding="utf-8-sig")
    subir(os.path.join(ruta, NOMBRE_DATOS), DRIVE_FOLDER_ID, datos_drive)
ctrl = pd.DataFrame([v["ctrl"] for v in leidos.values()] + control)
for c in ("AÑO", "MES", "DIAS", "DIAS_OK", "LOTES"):
    if c in ctrl:
        ctrl[c] = ctrl[c].astype("Int64")
ctrl["FECHA_EXTRACCION"] = date.today().strftime("%d/%m/%Y")
ctrl.to_csv(os.path.join(ruta, NOMBRE_CONTROL), index=False, encoding="utf-8-sig")
subir(os.path.join(ruta, NOMBRE_CONTROL), DRIVE_FOLDER_CONTROL_ID, control_drive)

# ── 7. RESUMEN ───────────────────────────────────────────────────────────────
print("═" * 70)
if len(ctrl):
    display(ctrl.drop(columns=["FECHA_EXTRACCION"]).sort_values(["AÑO", "MES", "FLUIDO"], na_position="first"))
if len(diaria):
    print(f"📅 Serie diaria: {diaria['FECHA'].nunique()} días · {len(diaria):,} filas · {NOMBRE_DATOS}")
    ult = diaria[["AÑO", "MES"]].drop_duplicates().sort_values(["AÑO", "MES"]).iloc[-1]
    m = diaria[(diaria["AÑO"] == ult["AÑO"]) & (diaria["MES"] == ult["MES"])]
    dias = m["DIA"].max()
    liq = m[m["UNIDAD"] == "BBL"].groupby(["PRODUCTO", "LOTE", "OPERADOR"])["VALOR"].sum() / dias
    gas = m[m["UNIDAD"] == "MCF"].groupby(["LOTE", "OPERADOR"])["VALOR"].sum() / dias / 1000
    print(f"Promedio diario de {MESES_ES[int(ult['MES']) - 1].lower()} {int(ult['AÑO'])} (días 1–{dias}):")
    if len(liq):
        display(liq[liq > 0].round(0).sort_values(ascending=False).to_frame("bpd"))
    if len(gas):
        display(gas[gas > 0].round(2).sort_values(ascending=False).to_frame("MMPCD"))
print(f"   Datos:   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
print(f"   Control: https://drive.google.com/drive/folders/{DRIVE_FOLDER_CONTROL_ID}")
